# Closed-loop inverse control: 1,000 steps

Run from `notebooks/` after generating the data and training the surrogate. Replay the final 1,000 chronological disturbances from the held-out test interval for three independent controllers: historical operator setpoints, Adam and LBFGS. Start each true plant from the same noise-free historical state at sample 6,999, then update first-order plant dynamics once per step and add the **same** seeded measurement-noise sequence to each controller. Each optimizer observes the current disturbances, keeps them fixed during optimization, and sends only its first optimized setpoint (receding horizon). The surrogate is static; it does not predict the dynamic state. The historical operator setpoints are replayed unchanged, even if they breach the normalized actuator limits.

Actuator commands are dimensionless, mapped to physical controls with training-only 1st/99th percentiles. The 0.05 rate limit and actuator bounds are hard-enforced for optimized controllers by sigmoid reparameterization. Predicted output limits and training-support limits use penalties and may be violated. Output thresholds are fixed using training-only output quantiles; no test outcomes are used to tune them. The economic score is a **unitless proxy**, not currency: standardized measured quality minus standardized measured energy minus 0.1 times squared control moves in input-standard-deviation units. Cumulative benefit is the sum of each optimizer's stepwise score minus the operator's score on the same disturbance/noise trajectory.

In [ ]:
from pathlib import Path
import ast
import pickle
import sys

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import torch
from scipy.special import expit
from torch import nn

sys.path.insert(0, str(Path('../src').resolve()))
from constraints import CONTROL_NAMES, SigmoidControlParameterization
from objective_functions import InverseProcessObjective

SEED = 42
N_STEPS = 1000
ADAM_STEPS = 20
LBFGS_STEPS = 20
POLICIES = ('Operator', 'Adam', 'LBFGS')
DATA_PATH = Path('../data/synthetic_process_data.csv')
MODEL_DIR = Path('../models')
RESULTS_DIR = Path('../results')
torch.manual_seed(SEED)
torch.set_num_threads(1)
checkpoint = torch.load(MODEL_DIR / 'process_model.pt', weights_only=True, map_location='cpu')
# Never load pickle scalers from an untrusted source.
with (MODEL_DIR / 'input_scaler.pkl').open('rb') as file:
    input_scaler = pickle.load(file)
with (MODEL_DIR / 'output_scaler.pkl').open('rb') as file:
    output_scaler = pickle.load(file)
input_columns, output_columns = checkpoint['input_columns'], checkpoint['output_columns']
if (len(input_columns) != 10 or input_columns[5:] != list(CONTROL_NAMES)
        or output_columns != ['y1_quality', 'y2_energy_load']
        or tuple(checkpoint['hidden_sizes']) != (32, 32, 16)):
    raise ValueError('Unexpected checkpoint architecture or column ordering')
if (list(input_scaler.feature_names_in_) != input_columns
        or list(output_scaler.feature_names_in_) != output_columns):
    raise ValueError('Saved scaler and checkpoint feature orders differ')

class ProcessModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 32), nn.SiLU(), nn.Linear(32, 32), nn.SiLU(),
            nn.Linear(32, 16), nn.SiLU(), nn.Linear(16, 2),
        )

    def forward(self, features):
        return self.network(features)

model = ProcessModel()
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval().requires_grad_(False)
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Run notebook 01 first: {DATA_PATH.resolve()}')
data = pd.read_csv(DATA_PATH)
if len(data) != 8000 or not np.isfinite(data[input_columns + output_columns].to_numpy(dtype=float)).all():
    raise ValueError('Expected 8,000 finite rows with all checkpoint features')
train = data.iloc[:5600]
if (not np.allclose(input_scaler.mean_, train[input_columns].mean().to_numpy(), rtol=1e-5)
        or not np.allclose(output_scaler.mean_, train[output_columns].mean().to_numpy(), rtol=1e-5)):
    raise ValueError('Scalers must match the first 70% of generated data')
start_index = len(data) - N_STEPS
if start_index < int(0.85 * len(data)):
    raise ValueError('The requested simulation horizon must fit within the held-out test interval')
trajectory = data.iloc[start_index:]
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Reuse exactly the original plant equation without rerunning data generation.
generator_path = Path('01_generate_data.ipynb')
generator = nbformat.read(generator_path, as_version=4)
definitions = [ast.parse(cell.source) for cell in generator.cells
               if cell.cell_type == 'code' and 'def steady_state_response(' in cell.source]
if len(definitions) != 1:
    raise ValueError('Original steady-state equation must occur in exactly one generator cell')
functions = [node for node in definitions[0].body
             if isinstance(node, ast.FunctionDef) and node.name == 'steady_state_response']
if len(functions) != 1:
    raise ValueError('Original plant response definition not found')
plant_scope = {'np': np, 'expit': expit}
exec(compile(ast.Module(body=functions, type_ignores=[]), str(generator_path), 'exec'), plant_scope)
plant_response = plant_scope['steady_state_response']
lag_gain = 1 - np.exp(-1 / np.array([8.0, 15.0]))
historical = data[input_columns].to_numpy()
historical_targets = plant_response(historical[:start_index, :5], historical[:start_index, 5:])
initial_plant_state = historical_targets[0].copy()
for target in historical_targets[1:]:
    initial_plant_state += lag_gain * (target - initial_plant_state)
measurement_noise = np.random.default_rng(SEED + 1).normal(
    0, [0.25, 0.35], size=(N_STEPS, 2))

support_lower = train[input_columns].quantile(0.01).to_numpy(dtype=np.float32)
support_upper = train[input_columns].quantile(0.99).to_numpy(dtype=np.float32)
command_origin = torch.from_numpy(support_lower[5:].copy())
command_span = torch.from_numpy((support_upper[5:] - support_lower[5:]).copy())
parameterization = SigmoidControlParameterization()
control_lower = command_origin + command_span * parameterization.lower
control_upper = command_origin + command_span * parameterization.upper
input_mean = torch.as_tensor(input_scaler.mean_, dtype=torch.float32)
input_scale = torch.as_tensor(input_scaler.scale_, dtype=torch.float32)
output_mean = torch.as_tensor(output_scaler.mean_, dtype=torch.float32)
output_scale = torch.as_tensor(output_scaler.scale_, dtype=torch.float32)
support_lower_tensor = torch.from_numpy(support_lower)
support_upper_tensor = torch.from_numpy(support_upper)
QUALITY_MIN = float(train['y1_quality'].quantile(0.25))
ENERGY_MAX = float(train['y2_energy_load'].quantile(0.75))
initial_controls = torch.from_numpy(historical[start_index - 1, 5:].astype(np.float32).copy())
initial_commands = (initial_controls - command_origin) / command_span
if not torch.all((initial_commands >= parameterization.lower)
                 & (initial_commands <= parameterization.upper)):
    raise ValueError('Prior historical setpoint is outside the normalized actuator limits')
print(f'Steps {start_index} through {len(data) - 1}; initial state {initial_plant_state}')
print(f'Fixed limits: quality >= {QUALITY_MIN:.3f}; energy <= {ENERGY_MAX:.3f}')

def predict(disturbances, controls):
    features = torch.cat((disturbances, controls))
    scaled = (features - input_mean) / input_scale
    return model(scaled.unsqueeze(0)).squeeze(0) * output_scale + output_mean

objective = InverseProcessObjective(
    model, input_mean, input_scale, output_mean, output_scale,
    control_lower, control_upper, support_lower_tensor, support_upper_tensor,
    quality_min=QUALITY_MIN, energy_max=ENERGY_MAX,
    quality_weight=1.0, energy_weight=1.0, movement_weight=0.1,
    constraint_weight=20.0, support_weight=10.0,
)


In [ ]:
def optimize_step(name, disturbances, previous_controls, previous_commands):
    """Choose one feasible command; previous controls remain fixed in the objective."""
    logits = nn.Parameter(torch.zeros(5))
    optimizer = (torch.optim.Adam([logits], lr=0.06) if name == 'Adam' else
                 torch.optim.LBFGS([logits], lr=0.8, max_iter=1,
                                   max_eval=10, line_search_fn='strong_wolfe'))
    iterations = ADAM_STEPS if name == 'Adam' else LBFGS_STEPS
    with torch.no_grad():
        best_loss = float(objective(disturbances, previous_controls, previous_controls))
        best_commands = previous_commands.clone()
    evaluations = 0
    for step in range(iterations):
        calls = [0]
        def closure():
            calls[0] += 1
            optimizer.zero_grad(set_to_none=True)
            commands = parameterization(logits, previous_commands)
            controls = command_origin + command_span * commands
            loss = objective(disturbances, controls, previous_controls)
            if not torch.isfinite(loss):
                raise ValueError(f'{name} has a non-finite loss at step {step}')
            loss.backward()
            if not torch.isfinite(logits.grad).all():
                raise ValueError(f'{name} has a non-finite gradient at step {step}')
            return loss
        if name == 'Adam':
            closure()
            optimizer.step()
        else:
            optimizer.step(closure)
        evaluations += calls[0]
        with torch.no_grad():
            candidate_commands = parameterization(logits, previous_commands)
            candidate_controls = command_origin + command_span * candidate_commands
            candidate_loss = float(objective(disturbances, candidate_controls, previous_controls))
            if not np.isfinite(candidate_loss):
                raise ValueError(f'{name} has a non-finite candidate loss at step {step}')
            if candidate_loss < best_loss:
                best_loss = candidate_loss
                best_commands = candidate_commands.clone()
    return best_commands, iterations, evaluations

def measure_violations(disturbances, controls, commands, previous_commands, outputs):
    full = torch.cat((disturbances, controls))
    return {
        'quality_shortfall': max(QUALITY_MIN - float(outputs[0]), 0.0),
        'energy_excess': max(float(outputs[1]) - ENERGY_MAX, 0.0),
        'bound_violation': float(torch.maximum(torch.relu(parameterization.lower - commands),
                                                torch.relu(commands - parameterization.upper)).max()),
        'rate_violation': float(torch.relu((commands - previous_commands).abs()
                                            - parameterization.max_move).max()),
        'support_violation': float(torch.maximum(
            torch.relu((support_lower_tensor - full) / input_scale),
            torch.relu((full - support_upper_tensor) / input_scale)).max()),
    }


In [ ]:
rows = []
states = {name: initial_plant_state.copy() for name in POLICIES}
last_controls = {name: initial_controls.clone() for name in POLICIES}
last_commands = {name: initial_commands.clone() for name in POLICIES}
for step, (_, record) in enumerate(trajectory.iterrows()):
    disturbances = torch.tensor(record[input_columns[:5]].to_numpy(dtype=np.float32))
    for name in POLICIES:
        previous_controls = last_controls[name]
        previous_commands = last_commands[name]
        if name == 'Operator':
            controls = torch.tensor(record[list(CONTROL_NAMES)].to_numpy(dtype=np.float32))
            commands = (controls - command_origin) / command_span
            iterations = evaluations = 0
        else:
            commands, iterations, evaluations = optimize_step(
                name, disturbances, previous_controls, previous_commands)
            controls = command_origin + command_span * commands
        with torch.no_grad():
            predicted = predict(disturbances, controls)
            loss = float(objective(disturbances, controls, previous_controls))
            static_target = plant_response(disturbances.numpy()[None, :],
                                           controls.numpy()[None, :])[0]
            states[name] = states[name] + lag_gain * (static_target - states[name])
            measured = states[name] + measurement_noise[step]
            command_move = commands - previous_commands
            physical_move = controls - previous_controls
            economic_score = ((measured[0] - output_mean[0].item()) / output_scale[0].item()
                              - (measured[1] - output_mean[1].item()) / output_scale[1].item()
                              - 0.1 * torch.sum((physical_move / input_scale[5:]) ** 2).item())
            predicted_violations = measure_violations(
                disturbances, controls, commands, previous_commands, predicted)
            observed_violations = {
                'observed_quality_shortfall': max(QUALITY_MIN - measured[0], 0.0),
                'observed_energy_excess': max(measured[1] - ENERGY_MAX, 0.0),
            }
            rows.append({
                'policy': name, 'step': step, 'sample_index': start_index + step,
                **{column: float(disturbances[i]) for i, column in enumerate(input_columns[:5])},
                **{column: float(controls[i]) for i, column in enumerate(CONTROL_NAMES)},
                'predicted_quality': float(predicted[0]), 'predicted_energy': float(predicted[1]),
                'true_quality': float(states[name][0]), 'true_energy': float(states[name][1]),
                'measured_quality': float(measured[0]), 'measured_energy': float(measured[1]),
                'objective': loss, 'economic_score': float(economic_score),
                'command_move_l2': float(torch.linalg.vector_norm(command_move)),
                'physical_move_l2': float(torch.linalg.vector_norm(physical_move)),
                'max_abs_command_move': float(command_move.abs().max()),
                'iterations': iterations, 'evaluations': evaluations,
                **predicted_violations, **observed_violations,
            })
        last_controls[name] = controls.detach()
        last_commands[name] = commands.detach()
rollout = pd.DataFrame(rows)
assert len(rollout) == N_STEPS * len(POLICIES)
if not np.isfinite(rollout.select_dtypes(include='number').to_numpy()).all():
    raise ValueError('Non-finite closed-loop simulation result')
rollout['cumulative_score'] = rollout.groupby('policy')['economic_score'].cumsum()
operator_score = rollout.loc[rollout.policy == 'Operator', 'economic_score'].to_numpy()
rollout['cumulative_benefit_vs_operator'] = rollout.groupby('policy', sort=False)['economic_score'].transform(
    lambda scores: np.cumsum(scores.to_numpy() - operator_score))
rollout.to_csv(RESULTS_DIR / 'closed_loop_trajectories.csv', index=False)
print(f'Exported {len(rollout)} policy-step observations')


In [ ]:
violation_columns = ['quality_shortfall', 'energy_excess', 'bound_violation',
                     'rate_violation', 'support_violation',
                     'observed_quality_shortfall', 'observed_energy_excess']
summary_rows = []
for name, frame in rollout.groupby('policy', sort=False):
    row = {
        'policy': name, 'steps': len(frame),
        'average_quality': frame.measured_quality.mean(),
        'average_energy': frame.measured_energy.mean(),
        'mean_quality_prediction_error': (frame.measured_quality - frame.predicted_quality).mean(),
        'mean_energy_prediction_error': (frame.measured_energy - frame.predicted_energy).mean(),
        'average_objective': frame.objective.mean(),
        'average_economic_score': frame.economic_score.mean(),
        'total_economic_score': frame.economic_score.sum(),
        'cumulative_benefit_vs_operator': frame.cumulative_benefit_vs_operator.iloc[-1],
        'mean_command_move_l2': frame.command_move_l2.mean(),
        'total_command_move_l2': frame.command_move_l2.sum(),
        'max_abs_command_move': frame.max_abs_command_move.max(),
        'total_optimizer_iterations': frame.iterations.sum(),
        'total_optimizer_evaluations': frame.evaluations.sum(),
        'any_predicted_violation_steps': (frame[violation_columns[:5]].max(axis=1) > 1e-5).sum(),
        'any_observed_output_violation_steps': (frame[violation_columns[5:]].max(axis=1) > 1e-5).sum(),
    }
    row.update({f'max_{column}': frame[column].max() for column in violation_columns})
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary.to_csv(RESULTS_DIR / 'closed_loop_summary.csv', index=False)
display(summary.T)
for row in summary.itertuples():
    if row.policy != 'Operator':
        print(f'{row.policy}: realized benefit vs operator = '
              f'{row.cumulative_benefit_vs_operator:.2f} unitless score points; '
              f'mean quality/energy prediction errors = '
              f'{row.mean_quality_prediction_error:.2f}/{row.mean_energy_prediction_error:.2f}')
        if row.cumulative_benefit_vs_operator < 0:
            print('  Warning: surrogate-optimal control underperforms historical operation '
                  'on the lagged true plant; do not deploy without model and control validation.')


In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(16, 12), sharex=True, constrained_layout=True)
for i, feature in enumerate(input_columns[:5]):
    axes[i].plot(trajectory.index, trajectory[feature], linewidth=0.7)
    axes[i].set_ylabel(feature)
axes[-1].set_xlabel('Sample index')
fig.suptitle('Shared disturbance trajectory')
fig.savefig(RESULTS_DIR / 'closed_loop_disturbances.png', dpi=150)
plt.show()

fig, axes = plt.subplots(5, 1, figsize=(16, 12), sharex=True, constrained_layout=True)
for name in POLICIES:
    frame = rollout[rollout.policy == name]
    for i, feature in enumerate(CONTROL_NAMES):
        axes[i].plot(frame.sample_index, frame[feature], label=name, linewidth=0.8, alpha=0.9)
for ax, feature in zip(axes, CONTROL_NAMES):
    ax.set_ylabel(feature)
axes[0].legend(ncol=3)
axes[-1].set_xlabel('Sample index')
fig.suptitle('Physical manipulated-variable setpoints')
fig.savefig(RESULTS_DIR / 'closed_loop_setpoints.png', dpi=150)
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True, constrained_layout=True)
for name in POLICIES:
    frame = rollout[rollout.policy == name]
    axes[0].plot(frame.sample_index, frame.measured_quality, label=name, linewidth=0.8)
    axes[1].plot(frame.sample_index, frame.measured_energy, label=name, linewidth=0.8)
axes[0].axhline(QUALITY_MIN, linestyle='--', color='black', label='Quality minimum')
axes[1].axhline(ENERGY_MAX, linestyle='--', color='black', label='Energy maximum')
axes[0].set_ylabel('Measured quality')
axes[1].set(xlabel='Sample index', ylabel='Measured energy')
axes[0].legend(ncol=4, fontsize=8)
axes[1].legend(fontsize=8)
fig.suptitle('Lagged, noisy true-process outputs')
fig.savefig(RESULTS_DIR / 'closed_loop_outputs.png', dpi=150)
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True, constrained_layout=True)
for name in POLICIES:
    frame = rollout[rollout.policy == name]
    axes[0].plot(frame.sample_index, frame.objective, label=name, linewidth=0.8)
    axes[1].plot(frame.sample_index, frame.cumulative_benefit_vs_operator, label=name, linewidth=1)
axes[0].set_ylabel('Penalized surrogate objective')
axes[1].set(xlabel='Sample index', ylabel='Cumulative economic benefit (score units)')
axes[0].legend(ncol=3)
axes[1].legend(ncol=3)
fig.suptitle('Controller objective and cumulative benefit')
fig.savefig(RESULTS_DIR / 'closed_loop_objective_benefit.png', dpi=150)
plt.show()
